# Clase 06: NLP con atencion usando Las mil y una noches

En NLP, resumir una secuencia larga en un unico vector suele ser un cuello de botella. La atencion aparece para relajar ese problema: en lugar de confiar en un resumen fijo, el modelo aprende a mirar que partes del contexto son mas utiles para la tarea.


In [ ]:
from collections import Counter
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

MODULE_ROOT = Path.cwd().resolve().parent
if str(MODULE_ROOT) not in sys.path:
    sys.path.append(str(MODULE_ROOT))

from tools.notebook_utils import choose_value, configure_runtime
from tools.text_corpus import build_vocabulary, chunk_text, corpus_stats, load_corpus_text, simple_tokenize

runtime = configure_runtime(seed=31)
print(runtime.summary())
print(corpus_stats())
        


## Por que existe la atencion

Una arquitectura encoder-decoder clasica intenta comprimir toda la entrada en un solo estado. Eso es costoso cuando el texto es largo, variado y contiene nombres, objetos y relaciones que reaparecen despues de muchas palabras.

La atencion existe para responder una pregunta puntual:

> si debo decidir algo ahora, sobre que partes del contexto me conviene apoyar la decision.

En esta notebook usamos ventanas largas del corpus para una tarea simple de clasificacion tematica. Las etiquetas se construyen con reglas debiles para que el foco este en el mecanismo, no en una annotation manual extensa.


In [ ]:
label_rules = {
    'corte': {'rey', 'palacio', 'visir', 'princesa', 'reina', 'trono'},
    'viaje': {'mar', 'isla', 'barco', 'mercader', 'nave', 'viaje'},
    'magia': {'genio', 'ifrit', 'jarra', 'anillo', 'espiritu', 'hechizo'},
}

raw_chunks = chunk_text(load_corpus_text(), chunk_size=55, overlap=20)
examples = []
for chunk in raw_chunks:
    tokens = simple_tokenize(chunk)
    scores = {label: sum(token in keywords for token in tokens) for label, keywords in label_rules.items()}
    label = max(scores, key=scores.get)
    if scores[label] >= 2:
        examples.append((tokens, label, chunk))

label_counts = Counter(label for _, label, _ in examples)
print({'usable_examples': len(examples), 'label_counts': dict(label_counts)})

label_to_id = {label: index for index, label in enumerate(sorted(label_rules))}
id_to_label = {index: label for label, index in label_to_id.items()}

train_examples, test_examples = train_test_split(
    examples,
    test_size=0.25,
    random_state=31,
    stratify=[label for _, label, _ in examples],
)
train_examples, val_examples = train_test_split(
    train_examples,
    test_size=0.25,
    random_state=31,
    stratify=[label for _, label, _ in train_examples],
)

train_tokens = [token for tokens, _, _ in train_examples for token in tokens]
vocab = build_vocabulary(train_tokens, min_freq=1)
print({'vocab_size': len(vocab.itos)})
        


In [ ]:
class AttentionDataset(Dataset):
    def __init__(self, items: list[tuple[list[str], str, str]]) -> None:
        self.items = items

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, int, str]:
        tokens, label, chunk = self.items[index]
        encoded = torch.tensor(vocab.encode(tokens), dtype=torch.long)
        return encoded, label_to_id[label], chunk


def collate_batch(batch):
    sequences, labels, raw_text = zip(*batch)
    lengths = torch.tensor([len(sequence) for sequence in sequences], dtype=torch.long)
    padded = pad_sequence(sequences, batch_first=True, padding_value=vocab.stoi['<pad>'])
    labels = torch.tensor(labels, dtype=torch.long)
    return padded, lengths, labels, list(raw_text)


train_loader = DataLoader(AttentionDataset(train_examples), batch_size=int(choose_value(16, 8)), shuffle=True, collate_fn=collate_batch)
val_loader = DataLoader(AttentionDataset(val_examples), batch_size=16, collate_fn=collate_batch)
test_loader = DataLoader(AttentionDataset(test_examples), batch_size=16, collate_fn=collate_batch)


class AdditiveAttention(torch.nn.Module):
    def __init__(self, hidden_dim: int) -> None:
        super().__init__()
        self.query_layer = torch.nn.Linear(hidden_dim, hidden_dim)
        self.key_layer = torch.nn.Linear(hidden_dim, hidden_dim)
        self.score_layer = torch.nn.Linear(hidden_dim, 1)

    def forward(self, hidden_states: torch.Tensor, mask: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        query = hidden_states.mean(dim=1, keepdim=True)
        energy = torch.tanh(self.query_layer(query) + self.key_layer(hidden_states))
        scores = self.score_layer(energy).squeeze(-1)
        scores = scores.masked_fill(~mask, -1e9)
        weights = torch.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), hidden_states).squeeze(1)
        return context, weights


class AttentionClassifier(torch.nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int = 48, hidden_dim: int = 64) -> None:
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embedding_dim, padding_idx=vocab.stoi['<pad>'])
        self.encoder = torch.nn.GRU(embedding_dim, hidden_dim // 2, batch_first=True, bidirectional=True)
        self.attention = AdditiveAttention(hidden_dim)
        self.classifier = torch.nn.Linear(hidden_dim, len(label_to_id))

    def forward(self, input_ids: torch.Tensor, lengths: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        embedded = self.embedding(input_ids)
        hidden_states, _ = self.encoder(embedded)
        mask = input_ids.ne(vocab.stoi['<pad>'])
        context, weights = self.attention(hidden_states, mask)
        logits = self.classifier(context)
        return logits, hidden_states, weights


model = AttentionClassifier(vocab_size=len(vocab.itos)).to(runtime.device)
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.005, weight_decay=1e-3)
        


In [ ]:
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
epochs = int(choose_value(20, 8))

for epoch in range(epochs):
    model.train()
    train_loss_total = 0.0
    for input_ids, lengths, labels, _ in train_loader:
        input_ids = input_ids.to(runtime.device)
        lengths = lengths.to(runtime.device)
        labels = labels.to(runtime.device)

        optimizer.zero_grad()
        logits, _, _ = model(input_ids, lengths)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        train_loss_total += loss.item() * len(input_ids)

    model.eval()
    val_loss_total = 0.0
    predictions = []
    targets = []
    with torch.inference_mode():
        for input_ids, lengths, labels, _ in val_loader:
            input_ids = input_ids.to(runtime.device)
            lengths = lengths.to(runtime.device)
            labels = labels.to(runtime.device)
            logits, _, _ = model(input_ids, lengths)
            val_loss_total += loss_fn(logits, labels).item() * len(input_ids)
            predictions.extend(logits.argmax(dim=1).cpu().tolist())
            targets.extend(labels.cpu().tolist())

    history['train_loss'].append(train_loss_total / len(train_loader.dataset))
    history['val_loss'].append(val_loss_total / len(val_loader.dataset))
    history['val_acc'].append(accuracy_score(targets, predictions))

plt.figure(figsize=(8, 3))
plt.plot(history['train_loss'], label='train_loss')
plt.plot(history['val_loss'], label='val_loss')
plt.title('Atencion sobre ventanas de texto: loss')
plt.legend()
plt.tight_layout()

plt.figure(figsize=(8, 3))
plt.plot(history['val_acc'], label='val_acc')
plt.title('Accuracy de validacion')
plt.legend()
plt.tight_layout()
        


In [ ]:
model.eval()
test_predictions = []
test_targets = []
first_batch = None
with torch.inference_mode():
    for batch in test_loader:
        if first_batch is None:
            first_batch = batch
        input_ids, lengths, labels, _ = batch
        logits, _, _ = model(input_ids.to(runtime.device), lengths.to(runtime.device))
        test_predictions.extend(logits.argmax(dim=1).cpu().tolist())
        test_targets.extend(labels.tolist())

print({'test_acc': round(accuracy_score(test_targets, test_predictions), 3)})

input_ids, lengths, labels, raw_text = first_batch
with torch.inference_mode():
    logits, hidden_states, additive_weights = model(input_ids.to(runtime.device), lengths.to(runtime.device))

sample_index = 0
sample_ids = input_ids[sample_index]
valid_mask = sample_ids.ne(vocab.stoi['<pad>'])
sample_tokens = vocab.decode(sample_ids[valid_mask].tolist())
query_vector = hidden_states[sample_index, valid_mask.to(runtime.device)].mean(dim=0)
dot_scores = torch.matmul(hidden_states[sample_index, valid_mask.to(runtime.device)], query_vector)
dot_weights = torch.softmax(dot_scores, dim=0).cpu()
additive = additive_weights[sample_index, valid_mask.to(runtime.device)].cpu()

ranking = sorted(
    zip(sample_tokens, additive.tolist(), dot_weights.tolist()),
    key=lambda item: item[1],
    reverse=True,
)[:10]

print('Texto de ejemplo:')
print(raw_text[sample_index][:300], '...')
print({'etiqueta_real': id_to_label[int(labels[sample_index])], 'etiqueta_predicha': id_to_label[int(logits[sample_index].argmax().cpu())]})
print('Top tokens por atencion aditiva y dot-product:')
for token, add_score, dot_score in ranking:
    print({'token': token, 'additive': round(add_score, 3), 'dot': round(dot_score, 3)})

plt.figure(figsize=(10, 3))
plt.bar(range(len(additive[:12])), additive[:12].tolist(), label='additive')
plt.plot(range(len(dot_weights[:12])), dot_weights[:12].tolist(), marker='o', label='dot-product')
plt.xticks(range(len(sample_tokens[:12])), sample_tokens[:12], rotation=45, ha='right')
plt.title('Comparacion de pesos de atencion sobre un fragmento')
plt.legend()
plt.tight_layout()
        


## Para cerrar

### Lo importante

- la atencion no reemplaza toda la arquitectura, pero cambia como se usa el contexto,
- una ventana larga no sirve de nada si el modelo no puede focalizar,
- los pesos de atencion no son una prueba causal, pero si una pista util para inspeccionar el comportamiento del modelo.

### Ejercicios sugeridos

- cambia las reglas de etiquetas y mira como cambia la interpretacion,
- aumenta `chunk_size` y observa si la atencion se vuelve mas selectiva,
- reemplaza la GRU por una BiLSTM y compara costo y accuracy.
